# 03 - Erro com pontos flutuantes
Vamos aprender sobre como usar os erros de ponto flutuante para resolução de problemas numéricos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana3
```

Faça o checkout nessa nova branch:

```bash
git checkout semana3
```

<hr />

## Atividade 1
A função exponencial natural pode ser definida pelo limite:
$$
e^x=\lim_{n\to\infty}\left(1+\frac{x}{n}\right)^n,
$$
mas também é dada pela **série de Maclaurin**:
$$
e^x=\sum_{n=0}^{\infty}\frac{x^n}{n!}
=1+\frac{x}{1!}+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

Implemente em **Python** o cálculo de $e^x$ pela série, interrompendo a soma quando o termo ficar menor que o limite prático de contribuição, usando a precisão de máquina como critério.

In [ ]:
import math
import sys

def exp_series(x, atol=0.0):
    """ Aproxima e^x pela série de Maclaurin com critério de parada numérico. """
    eps = sys.float_info.epsilon
    s = 1.0
    term = 1.0
    n = 0
    tol_abs = max(atol, eps)

    while True:
        n += 1
        term *= x / n
        s += term
        if abs(term) < eps * abs(s) or abs(term) < tol_abs:
            break
        if n > 10_000:
            break
    return s, n, term

for val in [1.0, 5.0, -2.0]:
    approx, nterms, last = exp_series(val)
    print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, termos={nterms})")

## Atividade 2

Implemente:
$$
e^x\approx\left(1+\frac{x}{n}\right)^n
$$
com $n$ crescente, e:
1. Explique por que, para $x<0$ e $n$ muito grande, pode ocorrer **cancelamento catastrófico**;
2. Proponha um critério de parada numérico para encerrar o crescimento de $n$ sem perder precisão.

In [ ]:
import math

def aproxima_exp(x, tol=1e-15):
    n = 1
    a = (1 + x/n)**n

    while True:
        n *= 2
        base = 1 + x/n

        if base == 1.0:
            return a, n

        novo = base**n
        erro = abs(novo - a) / max(1.0, abs(novo))

        if erro < tol:
            return novo, n

        a = novo

## Atividade 3

Para $|x|$ grande, use:
$$
e^x = \left(e^{m\cdot 2^{-k}}\right)^{2^k}, \quad
k = \left\lceil \log_2\!\left(\frac{|x|}{\theta}\right)\right\rceil, \quad m = \frac{x}{2^k}
$$
Calcule $e^{m}$ pela série (Ex. 1) e depois eleve ao quadrado $k$ vezes.

In [ ]:
import math

def exp_reduzido(x, theta=1.0, tol=1e-15):
    if x == 0:
        return 1.0

    k = math.ceil(math.log2(abs(x) / theta)) if abs(x) > theta else 0
    m = x / (2**k)

    termo = 1.0
    soma = 1.0
    j = 1

    while abs(termo) > tol:
        termo *= m / j
        soma += termo
        j += 1

    resultado = soma
    for _ in range(k):
        resultado *= resultado

    return resultado

## Atividade 4

Use:
$$
\cos x=\sum_{n=0}^{\infty}(-1)^n\frac{x^{2n}}{(2n)!}
$$
com a recursão:
$$
t_{n+1}=t_n\cdot\frac{-x^2}{(2n+1)(2n+2)}
$$
Defina um critério de parada baseado em `epsilon` e compare o erro relativo para $x\in[-20,20]$ (200 pontos) contra `math.cos(x)`.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def cos_aprox(x, epsilon=1e-15):
    t = 1.0
    soma = t
    n = 0

    while True:
        t = t * (-x**2) / ((2*n + 1)*(2*n + 2))
        soma_novo = soma + t

        if abs(t) <= epsilon * max(1.0, abs(soma_novo)):
            return soma_novo

        soma = soma_novo
        n += 1

x_values = np.linspace(-20, 20, 200)

erros_relativos = []

for x in x_values:
    valor_aprox = cos_aprox(x)
    valor_exato = math.cos(x)

    if valor_exato != 0:
        erro_relativo = abs(valor_aprox - valor_exato) / abs(valor_exato)
    else:
        erro_relativo = abs(valor_aprox - valor_exato)

    erros_relativos.append(erro_relativo)

erros_relativos = np.array(erros_relativos)

print("Erro relativo máximo:", np.max(erros_relativos))
print("Erro relativo médio:", np.mean(erros_relativos))

plt.figure(figsize=(10, 5))
plt.plot(x_values, erros_relativos)
plt.xlabel("x")
plt.ylabel("Erro relativo")
plt.title("Erro relativo da aproximação de cos(x)")
plt.yscale("log")
plt.grid(True)
plt.show()

## Atividade 5

Dado $x$ e uma tolerância $\tau$, encontre o menor $N$ tal que:
$$
R_{N+1}(x)=\sum_{n=N+1}^{\infty}\frac{|x|^n}{n!} < \tau
$$

In [ ]:
import math

def menor_N(x, tau):
    termo = 1.0
    soma = 0.0
    n = 0

    while True:
        soma += termo

        if n >= 1:
            resto = abs(x)**(n + 1) / math.factorial(n + 1)

            if resto * (1 + abs(x)/(n + 2)) < tau:
                return n

        n += 1
        termo *= abs(x) / n

## Atividade 6

Usando `decimal` ou `mpmath`, compute $e^x$ em alta precisão e compare com o resultado de `float64` (Ex. 1) para $x\in\{20, 40, 50\}$.
Analise:
- perda de dígitos significativos;
- quando o `float64` começa a saturar por overflow.


In [ ]:
import math
import mpmath as mp

mp.mp.dps = 80

def exp_float64(x, epsilon=1e-15):
    termo = 1.0
    soma = 1.0
    n = 1

    while True:
        termo *= x / n
        novo = soma + termo

        if abs(termo) <= epsilon * abs(novo):
            return novo

        soma = novo
        n += 1

def exp_alta_precisao(x):
    return mp.exp(mp.mpf(x))

valores = [20, 40, 50]

print(f"{'x':>5} {'float64':>25} {'mpmath':>25} {'erro relativo':>20} {'dígitos':>12}")

for x in valores:
    valor_float = exp_float64(x)
    valor_mp = exp_alta_precisao(x)

    erro = abs(mp.mpf(valor_float) - valor_mp) / abs(valor_mp)
    digitos = -mp.log10(erro) if erro != 0 else mp.inf

    print(
        f"{x:5d} "
        f"{valor_float:25.16e} "
        f"{mp.nstr(valor_mp, 18):>25} "
        f"{mp.nstr(erro, 8):>20} "
        f"{mp.nstr(digitos, 8):>12}"
    )

limite_overflow = math.log(sys.float_info.max) if 'sys' in globals() else 709.782712893384

print("\nLimite aproximado de overflow do float64:")
print(limite_overflow)

print("\nExponencial próxima ao limite:")
for x in [700, 709, 709.78, 710, 720]:
    try:
        print(x, math.exp(x))
    except OverflowError:
        print(x, "overflow")

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 3"
git push origin semana3
```